# WX · 00 · Build the matched D3 wMNE / Xu-protocol dataset
The same recordings, segments, five-minute selection, 10 s / 1 s windows, rejection, Xu bands, PSD / DE / PLV /
|ciPLV|, 68 Desikan-Killiany ROIs, mode A (tenfold) and mode B as the corrected LCMV stores; **only the inverse
differs**: the historical wMNE settings (depth-weighted MNE, loose 0.2, lambda2 1/9, ad-hoc noise covariance,
normal component) instead of LCMV. One data-independent 68 x 19 operator for the shared electrode geometry,
verified against MNE's `apply_inverse_raw` on every participant.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).
**Steps:** 1. input check · 2. recording QC (no source model) · 3. smoke test, one participant (`_limit1` run) ·
4. full build (only if the smoke test passed; resumable per participant) · 5. validation of both stores (exact
boundary positions, 300 s cap, split, shapes, weights, counts) · 6. **match** against the corrected LCMV stores:
identical participants, labels, selections, window ids and start / stop samples, tenfold flags, weights, split,
ROI order and sensor features (the Exp 0 consistency control); experiments refuse stores that do not match ·
7. QC report. CPU runtime (no GPU needed).

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

## 1. Input check (36 AD / 29 HC, 19 channels and verified electrode coordinates)

In [ ]:
sh("python scripts/d3_data.py check")

## 2. Recording QC of every participant (no source model)

In [ ]:
sh("python scripts/wx_build.py inspect")

## 3. Smoke test: one participant, full wMNE path (separate `_limit1` run)

In [ ]:
sh("python scripts/wx_build.py build --limit 1")
sh("python scripts/wx_build.py validate --limit 1")
SMOKE_PASSED = True                       # reached only if both commands above succeeded

## 4. Full build (all 65 participants; resumable, completed participants are skipped)

In [ ]:
if not SMOKE_PASSED:
    raise RuntimeError('smoke test did not pass; the full build is not started')
sh("python scripts/wx_build.py build")

## 5. Validation of both stores

In [ ]:
sh("python scripts/wx_build.py validate")

## 6. Match against the corrected LCMV stores (read only)

In [ ]:
sh("python scripts/wx_build.py match")

## 7. QC report

In [ ]:
import glob, json
import pandas as pd
run = sorted(p for p in glob.glob(WX + '/graphsWX_v*') if '_limit' not in p)[-1]
qc = pd.read_csv(run + '/qc_report.csv')
pd.set_option('display.max_rows', 100, 'display.width', 250)
display(qc[[c for c in ['key', 'label', 'selection_mode', 'selected_duration_s', 'n_windows_candidate',
                        'n_windows_accepted', 'n_tenfold', 'wmne_verification_rel_error', 'status', 'reason'] if c in qc]])
print(qc['status'].value_counts().to_dict(), '| windows (mode B):', int(qc['n_windows_kept'].sum()),
      '| samples (mode A):', int(qc['n_tenfold'].sum()))
for d in (run, run.replace('graphsWX_', 'graphsWX10_')):
    m = json.load(open(d + '/match.json'))
    print(d.split('/')[-1], 'matched to', m['reference'].split('/')[-1], '->', 'PASS' if m['passed'] else 'FAIL')